# Part 1 - Linear Regression from Scratch
**Gray Interface '26 | Task 2**

## Importing the dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Generating the Dataset

In [ ]:
# Fix random seed so results are reproducible
np.random.seed(42)

# Generate 100 samples: true relationship is y = 4 + 3X + noise
X = 2 * np.random.rand(100, 1)
noise = np.random.randn(100, 1)
y = 4 + 3 * X + noise

# Visualize the raw data
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='steelblue', alpha=0.7)
plt.title('Generated Dataset')
plt.xlabel('X')
plt.ylabel('y')
plt.tight_layout()
plt.show()


## Train-Test Split

In [ ]:
# Shuffle and split: 80 samples for training, 20 for testing
indices = np.random.permutation(100)
train_idx, test_idx = indices[:80], indices[80:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


## Linear Regression using Gradient Descent

In [ ]:
# Add a column of 1s so the model can learn the intercept (bias term)
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

X_train_b = add_bias(X_train)
X_test_b  = add_bias(X_test)


In [ ]:
def compute_loss(X_b, y, weights):
    # Mean Squared Error: average of (prediction - actual)^2
    m = len(y)
    predictions = X_b @ weights
    return (1 / (2 * m)) * np.sum((predictions - y) ** 2)

def gradient_descent(X_b, y, lr=0.1, epochs=1000):
    m = len(y)
    # Start with weights = 0
    weights = np.zeros((X_b.shape[1], 1))
    loss_history = []

    for epoch in range(epochs):
        predictions = X_b @ weights

        # Gradient tells us which direction loss increases
        # We move in the opposite direction to reduce loss
        gradient = (1 / m) * X_b.T @ (predictions - y)
        weights = weights - lr * gradient

        # Save loss every 10 steps to plot later
        if epoch % 10 == 0:
            loss_history.append(compute_loss(X_b, y, weights))

    return weights, loss_history


## Experimenting with Different Learning Rates

In [ ]:
# A high lr converges fast but can overshoot; a low lr is stable but slow
learning_rates = [0.01, 0.1, 0.5]
epochs = 200

plt.figure(figsize=(10, 5))
for lr in learning_rates:
    _, loss_hist = gradient_descent(X_train_b, y_train, lr=lr, epochs=epochs)
    plt.plot(range(0, epochs, 10), loss_hist, label=f'lr = {lr}')

plt.title('Loss vs Epochs for Different Learning Rates')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.tight_layout()
plt.show()


## Training the Final Model

In [ ]:
# lr=0.1 gave the best balance of speed and stability
weights, loss_history = gradient_descent(X_train_b, y_train, lr=0.1, epochs=1000)

print(f"Learned intercept (w0): {weights[0][0]:.4f}  (true value: 4)")
print(f"Learned slope     (w1): {weights[1][0]:.4f}  (true value: 3)")


## Loss vs Epochs

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(0, 1000, 10), loss_history, color='#E50914')
plt.title('Loss vs Epochs (lr=0.1, epochs=1000)')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.tight_layout()
plt.show()


## Visualizing the Fitted Line

In [ ]:
X_line = np.array([[0], [2]])
y_line = add_bias(X_line) @ weights

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='steelblue', alpha=0.6, label='Train')
plt.scatter(X_test,  y_test,  color='orange',    alpha=0.6, label='Test')
plt.plot(X_line, y_line, color='red', linewidth=2, label='Fitted line')
plt.title('Linear Regression - Fitted Line')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.tight_layout()
plt.show()


## Evaluation Metrics

In [ ]:
def evaluate(X_b, y, weights, label):
    preds = X_b @ weights
    mae   = np.mean(np.abs(preds - y))
    mse   = np.mean((preds - y) ** 2)
    rmse  = np.sqrt(mse)
    r2    = 1 - np.sum((preds - y)**2) / np.sum((y - np.mean(y))**2)

    print(f"--- {label} ---")
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")
    print()

evaluate(X_train_b, y_train, weights, "Training Set")
evaluate(X_test_b,  y_test,  weights, "Test Set")


## Bonus: Polynomial Regression (Degree 2)

In [ ]:
# Add X^2 as an extra feature to capture non-linear patterns
def poly_features(X):
    return np.hstack([np.ones((X.shape[0], 1)), X, X**2])

X_train_p = poly_features(X_train)
X_test_p  = poly_features(X_test)

weights_poly, _ = gradient_descent(X_train_p, y_train, lr=0.05, epochs=2000)
print("Polynomial weights:", weights_poly.ravel())


In [ ]:
# Compare both models visually
X_plot = np.linspace(0, 2, 200).reshape(-1, 1)
y_lin  = add_bias(X_plot) @ weights
y_poly = poly_features(X_plot) @ weights_poly

plt.figure(figsize=(9, 5))
plt.scatter(X, y, color='steelblue', alpha=0.5, label='Data')
plt.plot(X_plot, y_lin,  color='red',   linewidth=2,              label='Linear')
plt.plot(X_plot, y_poly, color='green', linewidth=2, linestyle='--', label='Polynomial deg=2')
plt.title('Linear vs Polynomial Regression')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Compare metrics side by side
evaluate(X_train_b, y_train, weights,      "Linear     - Train")
evaluate(X_test_b,  y_test,  weights,      "Linear     - Test")
evaluate(X_train_p, y_train, weights_poly, "Polynomial - Train")
evaluate(X_test_p,  y_test,  weights_poly, "Polynomial - Test")


## Observations

- **lr = 0.01**: Very slow convergence — loss still dropping at epoch 200.
- **lr = 0.1**: Smooth convergence, reaches minimum around epoch 150. Best choice.
- **lr = 0.5**: Converges fast early but risks overshooting the minimum on noisy data.
- The final model recovers weights very close to the true values (intercept ≈ 4, slope ≈ 3), confirming our gradient descent implementation is correct.
- **Polynomial degree 2** gets marginally better training metrics but nearly identical test metrics — since the true relationship is linear, the extra X² term doesn't help generalize.
